In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import ta  # technical analysis indicators: RSI, MACD, Bollinger Bands, ATR

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Load the shared 5-minute intraday snapshot -- the same source file used in
# notebooks/exploration.ipynb. We reuse it instead of calling yfinance again so
# every notebook works from one already quality-checked dataset.
cwd = Path.cwd().resolve()
project_root = cwd.parent if cwd.name == "notebooks" else cwd
file_path = project_root / "data" / "raw" / "stock-trend.parquet"

df_saved = pd.read_parquet(file_path)

# Keep AAPL only and drop the now-redundant "Ticker" column level, so columns
# become simple names: Open, High, Low, Close, Volume.
df = df_saved.xs("AAPL", axis=1, level="Ticker").copy()

print(f"AAPL data shape: {df.shape}")
print(f"Period: {df.index.min()} to {df.index.max()}")
df.head()

# Feature Engineering — AAPL

Builds a per-5-minute-bar feature and target table for AAPL from the same intraday snapshot used in `exploration.ipynb`. As noted there: feature and target generation belongs here / in `src/features.py`, chronological splitting and evaluation belong in `src/train.py`, and future prices must never enter model inputs.

1. Candle-shape features (range, body)
2. Lagged return features (5m / 15m / 30m), gap-masked
3. Rolling volatility and relative volume, grouped by trading day
4. Technical indicators (RSI, MACD, Bollinger Band width, ATR)
5. Target: 30-minute-ahead direction and return, gap-masked
6. Assemble, drop warm-up/tail NaNs, and persist to `data/processed/aapl_training.parquet`

In [ ]:
# --- Candle-shape features ---
# range_pct: full high-low range of the candle, relative to the close price.
#   Larger values indicate higher intra-bar volatility.
# body_pct: open-to-close move, relative to the open price.
#   Sign shows candle direction (bullish/bearish); magnitude shows conviction.
df["range_pct"] = (df["High"] - df["Low"]) / df["Close"] * 100
df["body_pct"] = (df["Close"] - df["Open"]) / df["Open"] * 100

df[["range_pct", "body_pct"]].describe()

In [ ]:
# --- Lagged return features ---
def masked_pct_return(close: pd.Series, bars: int) -> pd.Series:
    """
    Percentage return over `bars` consecutive 5-minute candles.
    Returns NaN when the lookback window crosses a session gap (e.g. the last
    bars of one day into the first bars of the next), the same way
    exploration.ipynb masks returns to a strict 5-minute spacing -- this stops
    an overnight jump from being reported as a 5/15/30-minute move.
    """
    pct = close.pct_change(periods=bars, fill_method=None) * 100
    expected_start_time = close.index - pd.Timedelta(minutes=5 * bars)
    actual_start_time = close.index.to_series().shift(bars)
    is_contiguous = actual_start_time == expected_start_time
    return pct.where(is_contiguous)

df["return_5m_pct"] = masked_pct_return(df["Close"], bars=1)
df["return_15m_pct"] = masked_pct_return(df["Close"], bars=3)
df["return_30m_pct"] = masked_pct_return(df["Close"], bars=6)

df[["return_5m_pct", "return_15m_pct", "return_30m_pct"]].describe()

In [ ]:
# --- Rolling volatility and volume features ---
# Bars are grouped by calendar day first, so a rolling window never blends
# yesterday's last few candles with today's first few candles.
trading_day = df.index.normalize()

# Rolling std-dev of 5-minute returns over the trailing 6 bars (~30 minutes):
# a short-horizon realized-volatility measure.
df["volatility_30m"] = (
    df.groupby(trading_day)["return_5m_pct"]
    .rolling(window=6, min_periods=6)
    .std()
    .reset_index(level=0, drop=True)
)

# Volume relative to its trailing 20-bar (~100 minute) average for that day.
# Values above 1 flag unusually heavy trading for that time of day.
rolling_avg_volume = (
    df.groupby(trading_day)["Volume"]
    .rolling(window=20, min_periods=20)
    .mean()
    .reset_index(level=0, drop=True)
)
df["volume_relative"] = df["Volume"] / rolling_avg_volume

df[["volatility_30m", "volume_relative"]].describe()

In [ ]:
# --- Technical indicators (ta library) ---
# Computed on the raw, gap-inclusive Close/High/Low series -- this matches how
# these indicators are used in practice (they are running averages of price
# history, not bar-to-bar returns), but it does mean the first indicator value
# after an overnight gap still incorporates the prior session's data.
rsi = ta.momentum.RSIIndicator(close=df["Close"], window=14)
df["rsi_14"] = rsi.rsi()  # 0-100 momentum oscillator; >70 overbought, <30 oversold

macd = ta.trend.MACD(close=df["Close"])
df["macd_diff"] = macd.macd_diff()  # MACD line minus its signal line (momentum shift)

bb = ta.volatility.BollingerBands(close=df["Close"], window=20, window_dev=2)
df["bb_width_pct"] = (bb.bollinger_hband() - bb.bollinger_lband()) / df["Close"] * 100

atr = ta.volatility.AverageTrueRange(high=df["High"], low=df["Low"], close=df["Close"], window=14)
df["atr_pct"] = atr.average_true_range() / df["Close"] * 100  # ATR normalized by price

df[["rsi_14", "macd_diff", "bb_width_pct", "atr_pct"]].describe()

In [ ]:
# --- Target: will price be higher 30 minutes (6 bars) from now? ---
# Same gap-safety idea as the feature returns above: the label for a bar is only
# kept when the timestamp 30 minutes ahead is an actual observed bar in the data,
# not a slot that only exists because we skipped over an overnight/weekend gap.
horizon_bars = 6  # 6 bars * 5 minutes = 30 minutes ahead

future_close = df["Close"].shift(-horizon_bars)
future_time = df.index.to_series().shift(-horizon_bars)
expected_future_time = df.index + pd.Timedelta(minutes=5 * horizon_bars)
horizon_is_valid = future_time == expected_future_time

# Continuous target: forward return, useful for a regression model.
df["target_return_30m_pct"] = (
    (future_close - df["Close"]) / df["Close"] * 100
).where(horizon_is_valid)

# Binary target: direction only, useful for a classification model.
# `.where(horizon_is_valid)` is applied explicitly (not inferred from the NaN in
# target_return_30m_pct) because `np.nan > 0` evaluates to False, not NaN, and
# would otherwise mislabel invalid rows as "down" instead of "unknown".
df["target_up_30m"] = (df["target_return_30m_pct"] > 0).astype("Int64").where(horizon_is_valid)

# Kept for traceability: the exact timestamp each target refers to.
df["target_time"] = future_time.where(horizon_is_valid)

df[["target_return_30m_pct", "target_up_30m", "target_time"]].tail(10)

In [ ]:
# --- Assemble the final training table ---
feature_cols = [
    "range_pct", "body_pct",
    "return_5m_pct", "return_15m_pct", "return_30m_pct",
    "volatility_30m", "volume_relative",
    "rsi_14", "macd_diff", "bb_width_pct", "atr_pct",
]
target_cols = ["target_return_30m_pct", "target_up_30m", "target_time"]

df_features = df[feature_cols + target_cols].copy()

# Drop rows with missing values. NaNs come from two sources only:
#   1. Indicator/rolling warm-up at the start of the series (not enough history yet).
#   2. The last `horizon_bars` rows, which have no future bar to compute a target from.
# The DatetimeIndex keeps everything in chronological order throughout, so this
# never shuffles data and features are never computed from future information.
rows_before = len(df_features)
df_features = df_features.dropna()
print(f"Dropped {rows_before - len(df_features)} rows with missing feature/target values")
print(f"Final shape: {df_features.shape}")

df_features.head()

In [ ]:
# Persist the training table for src/train.py to consume.
# Saved as Parquet (not CSV) to preserve dtypes and the tz-aware DatetimeIndex.
processed_path = project_root / "data" / "processed" / "aapl_training.parquet"
processed_path.parent.mkdir(parents=True, exist_ok=True)
df_features.to_parquet(processed_path)

print(f"Saved {df_features.shape[0]} rows x {df_features.shape[1]} columns to {processed_path}")

## 6. Sanity Checks

In [ ]:
# Class balance of the target -- important to know before training, since a
# heavily imbalanced target (e.g. 90% "up") would make plain accuracy misleading.
df_features["target_up_30m"].value_counts(normalize=True).rename("share")